In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("multimodal_train.tsv", sep="\t")

In [3]:
#Analyze the dataset
df.head()
print("shape" , df.shape)
print("\nColumns name :")
print(df.columns.tolist())


shape (564000, 16)

Columns name :
['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id', 'image_url', 'linked_submission_id', 'num_comments', 'score', 'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label', '6_way_label']


In [4]:
print("\nLabel distribution:")
print(df["2_way_label"].value_counts())

print("\nImage availability:")
print(df["hasImage"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())


Label distribution:
2_way_label
0    341919
1    222081
Name: count, dtype: int64

Image availability:
hasImage
True    564000
Name: count, dtype: int64

Missing values:
author                   28710
clean_title                  0
created_utc                  0
domain                  167857
hasImage                     0
id                           0
image_url                 1534
linked_submission_id    396143
num_comments            167857
score                        0
subreddit                    0
title                        0
upvote_ratio            167857
2_way_label                  0
3_way_label                  0
6_way_label                  0
dtype: int64


# now clean the dataset 

In [5]:
# Keep only columns needed initially
df = df[
    [
       "id",
        "clean_title",
        "image_url",
        "domain",
        "subreddit",
        "created_utc",
        "2_way_label"   
    ]
  ]

#now we remove the rows without image urls 

df = df.dropna(subset = ["image_url"])

# remove the duplicate ids
df = df.drop_duplicates(subset = ["id"])

print("After cleaning:", df.shape)
print("\nLabels:")
print(df["2_way_label"].value_counts())

After cleaning: (562466, 7)

Labels:
2_way_label
0    341460
1    221006
Name: count, dtype: int64


In [6]:
# now we select the 10,000 smaples of 2_way_label each 0 , 1 

# for 0 label
df_0 = df[df["2_way_label"] == 0].sample(
    n=10000 , random_state = 42
)

# for 1 label
df_1 = df[df["2_way_label"] == 1].sample(
    n=10000, random_state=42
)

# now combine the both clases

df_small = pd.concat([df_0 , df_1])

# now shuffle the dataset for random 0 and 1 

df_small = df_small.sample(
    frac=1 , random_state=42
).reset_index(drop = True)

print("Dataset shape:", df_small.shape)

print("\nLabel distribution:")
print(df_small["2_way_label"].value_counts())

Dataset shape: (20000, 7)

Label distribution:
2_way_label
1    10000
0    10000
Name: count, dtype: int64


In [7]:
# now save the dataset 

import os

os.makedirs("data/processed", exist_ok=True)

df_small.to_csv(
    "data/processed/train_20k.csv",
    index=False
)

print("Dataset saved successfully!")

Dataset saved successfully!


In [8]:
# now we can take the validation data 
val_df = pd.read_csv(
    "multimodal_validate.tsv",
    sep="\t"
)

print("Validation shape:", val_df.shape)
print("\nColumns:")
print(val_df.columns.tolist())

print("\nLabels:")
print(val_df["2_way_label"].value_counts())

print("\nImages:")
print(val_df["hasImage"].value_counts())

Validation shape: (59342, 16)

Columns:
['author', 'clean_title', 'created_utc', 'domain', 'hasImage', 'id', 'image_url', 'linked_submission_id', 'num_comments', 'score', 'subreddit', 'title', 'upvote_ratio', '2_way_label', '3_way_label', '6_way_label']

Labels:
2_way_label
0    36022
1    23320
Name: count, dtype: int64

Images:
hasImage
True    59342
Name: count, dtype: int64


In [9]:
#now we can take the test data
test_df = pd.read_csv(
    "multimodal_test_public.tsv",
    sep="\t"
)

print("Test shape:", test_df.shape)
print("\nLabels:")
print(test_df["2_way_label"].value_counts())

print("\nImages:")
print(test_df["hasImage"].value_counts())

Test shape: (59319, 16)

Labels:
2_way_label
0    35812
1    23507
Name: count, dtype: int64

Images:
hasImage
True    59319
Name: count, dtype: int64


In [10]:
# for validation data 

# Select 5,000 samples from each class
val_0 = val_df[val_df["2_way_label"] == 0].sample(
    n=5000,
    random_state=42
)

val_1 = val_df[val_df["2_way_label"] == 1].sample(
    n=5000,
    random_state=42
)

# Combine and shuffle
val_small = pd.concat([val_0, val_1])

val_small = val_small.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Validation shape:", val_small.shape)
print("\nLabel distribution:")
print(val_small["2_way_label"].value_counts())

Validation shape: (10000, 16)

Label distribution:
2_way_label
1    5000
0    5000
Name: count, dtype: int64


In [11]:
# for test data 

# Select 5,000 samples from each class
test_0 = test_df[test_df["2_way_label"] == 0].sample(
    n=5000,
    random_state=42
)

test_1 = test_df[test_df["2_way_label"] == 1].sample(
    n=5000,
    random_state=42
)

# Combine and shuffle
test_small = pd.concat([test_0, test_1])

test_small = test_small.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Test shape:", test_small.shape)
print("\nLabel distribution:")
print(test_small["2_way_label"].value_counts())

Test shape: (10000, 16)

Label distribution:
2_way_label
1    5000
0    5000
Name: count, dtype: int64


In [12]:
#now save the data 
os.makedirs("data/processed", exist_ok=True)

val_small.to_csv(
    "data/processed/validation_10k.csv",
    index=False
)

test_small.to_csv(
    "data/processed/test_10k.csv",
    index=False
)

print("Validation and test datasets saved!")

Validation and test datasets saved!


In [13]:
# check image urls 

print("Missing image URLs:")
print(df_small["image_url"].isna().sum())

print("\nExample image URLs:")
print(df_small["image_url"].head(10).to_string(index=False))

Missing image URLs:
0

Example image URLs:
https://preview.redd.it/ey5rao0mcyl01.jpg?width...
https://external-preview.redd.it/2Mz83vAFfUwLqc...
                    http://i.imgur.com/Q85pUjI.jpg
https://preview.redd.it/v88frif6y1g31.jpg?width...
https://external-preview.redd.it/bcm7FU6fOZ9qHS...
https://external-preview.redd.it/5Ni__0j8HTOln1...
https://preview.redd.it/8qvghw8h1gv21.jpg?width...
https://preview.redd.it/3k581ko30xb11.jpg?width...
https://preview.redd.it/2wpn68lh2vn11.jpg?width...
https://external-preview.redd.it/BZB6tKAVFO1zer...
